In [3]:
import requests

url = "http://localhost:11434/api/generate"

payload = {
    "model": "llama3.1:8b",
    "prompt": "Explain RAG in simple terms : in one line",
    "stream": False
}

response = requests.post(url, json=payload)

print(response.json()["response"])

RAG is a method for categorizing and managing data by dividing it into three colors: Red (problems or errors), Amber ( warnings or potential issues), and Green (good or acceptable).


In [ ]:
import os
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.1:8b",
    base_url="http://localhost:11434"
)

data = llm.invoke("Explain RAG in simple terms: in one line")

print(data)

In [ ]:
import os
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

# 1. Initialize your working local Ollama model
llm = OllamaLLM(model="llama3.1:8b", base_url="http://localhost:11434")

# 2. Define your prompt template
prompt_template = PromptTemplate.from_template(
    "Give me a list of {number} famous dishes from {cuisine} cuisine."
)

# 3. Create the chain using LCEL (The pipe '|' operator)
# This automatically feeds the formatted prompt directly into your Llama 3.1 model
chain = prompt_template | llm

# 4. Invoke the chain
response = chain.invoke({"cuisine": "Italian", "number": 3})

print(response)

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Connect to your working local Ollama server
llm = OllamaLLM(model="llama3.1:8b", base_url="http://localhost:11434")

# 2. STEP 1: Define Prompt & Chain to get a restaurant name
prompt1 = PromptTemplate.from_template(
    "You are a food guide. Name exactly one highly-rated, famous restaurant in {country}. "
    "Provide only the name of the restaurant, nothing else."
)
# We add StrOutputParser() to ensure the output is a clean string, not an object
chain1 = prompt1 | llm | StrOutputParser()


# 3. STEP 2: Define Prompt & Chain to get the dishes
prompt2 = PromptTemplate.from_template(
    "The user wants to visit the restaurant '{restaurant}' in {country}. "
    "Provide a sequential list of exactly 3 famous dishes to try there."
)
chain2 = prompt2 | llm


# 4. SEQUENTIAL CHAIN: Combine Step 1 and Step 2
# We use a dictionary to dynamically pass the original 'country' AND the 
# generated 'restaurant' from chain1 into chain2.
sequential_chain = (
    {"country": RunnablePassthrough(), "restaurant": chain1}
    | chain2
)

# 5. Run the sequential conversation
country_input = "Italy"
print(f"--- Processing requests for: {country_input} ---\n")

# This triggers the entire sequence seamlessly
final_response = sequential_chain.invoke(country_input)

print(final_response)

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Connect to your local Ollama server
llm = OllamaLLM(model="llama3.1:8b", base_url="http://localhost:11434")

# 2. STEP 1: Find a restaurant based on Country and Food Type
prompt1 = PromptTemplate.from_template(
    "You are a local food expert. Suggest exactly one highly-rated restaurant or spot "
    "in {country} that is famous for {food_type} options. "
    "Provide ONLY the name of the restaurant, nothing else."
)
chain1 = prompt1 | llm | StrOutputParser()


# 3. STEP 2: Get a dynamic NUMBER of recommendations
prompt2 = PromptTemplate.from_template(
    "The user wants to visit '{restaurant}' in {country} specifically for their {food_type} options. "
    "Provide a sequential list of exactly {number} specific dishes, drinks, or items they should order there."
)
chain2 = prompt2 | llm


# 4. SEQUENTIAL CHAIN: Pass all 3 initial inputs + the generated restaurant to Step 2
sequential_chain = (
    {
        "country": lambda x: x["country"],
        "food_type": lambda x: x["food_type"],
        "number": lambda x: x["number"],        # <-- Captures your dynamic number input
        "restaurant": chain1                     # <-- Captures Llama's generated output
    }
    | chain2
)

# 5. Execute with your dynamic inputs
user_inputs = {
    "country": "France",
    "food_type": "Dessert",
    "number": 4  # <-- Change this to 2, 3, 5, etc. from your side!
}

print(f"--- Requesting {user_inputs['number']} {user_inputs['food_type']} options in {user_inputs['country']} ---\n")

final_response = sequential_chain.invoke(user_inputs)
print(final_response)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent

# 1. LLM (Ollama local model)
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.7
)

# 2. Wikipedia Tool
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
tools = [wiki]

# 3. Create the Agent 
# This builds an automatic ReAct loop behind the scenes.
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use your tools to answer questions precisely."
)

# 4. Ask your question using the modern message structure
try:
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": "Who is Nikola Tesla?"}
        ]
    })
    
    # The output contains a list of messages; we print the final answer
    print(response["messages"][-1].content)

except Exception as e:
    print(f"Error: {e}")

In [ ]:
import sys
from langchain_ollama import ChatOllama
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# ==========================================
# 1. LLM Setup
# ==========================================
print("🔄 Initializing Ollama (Llama 3.1)...")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60  
)

# ==========================================
# 2. Tools Configuration
# ==========================================
print("📦 Loading search and math tools...")
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Use this tool for math calculations like 2+2, 10*5, etc."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if not all(char in allowed_chars for char in expression):
            return "Error: Invalid characters in math expression."
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

tools = [wiki, search_tool, calculator]

# ==========================================
# 3. Create Modern Agent with Checkpointer
# ==========================================
print("🤖 Compiling ReAct Agent Graph...")

# Using InMemorySaver handles conversation thread states automatically
memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. Think step-by-step. "
        "Use your search, wikipedia, and calculator tools whenever you need external or precise information."
    ),
    checkpointer=memory,
    debug=False  # Toggle to True if you want to inspect low-level LangGraph execution traces
)

# Configuration for managing conversation thread ID
config = {"configurable": {"thread_id": "jupyter_chat_session"}}

# ==========================================
# 4. Interactive Jupyter Function
# ==========================================
def ask_agent(user_input: str):
    """Call this function inside any Jupyter cell to talk to the agent."""
    if not user_input.strip():
        return
        
    print(f"You: {user_input}")
    print("Thinking...")
    
    try:
        # Pass the message payload structure expected by create_agent
        result = agent.invoke(
            {"messages": [{"role": "user", "content": user_input}]}, 
            config=config
        )
        
        # Pull the absolute last response generated in the message graph state
        ai_response = result["messages"][-1].content
        print(f"\nAI: {ai_response}\n")
        
    except Exception as e:
        print(f"\nExecution Error: {e}\n")

print("\n🚀 Setup Complete! You can now converse with the agent using the ask_agent() function.")

In [ ]:
ask_agent("Who was Nikola Tesla?")

In [1]:
import sys
from langchain_ollama import ChatOllama
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

# ==========================================
# 1. LLM Setup
# ==========================================
print("🔄 Initializing Ollama (Llama 3.1)...")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60  
)

# ==========================================
# 2. Tools Configuration
# ==========================================
print("📦 Loading search and math tools...")
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Use this tool for math calculations like 2+2, 10*5, etc."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if not all(char in allowed_chars for char in expression):
            return "Error: Invalid characters in math expression."
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

tools = [wiki, search_tool, calculator]

# ==========================================
# 3. Create Modern ReAct Agent with Checkpointer
# ==========================================
print("🤖 Compiling ReAct Agent Graph...")

# Using InMemorySaver handles conversation thread states automatically
memory = InMemorySaver()

# Create a proper prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful assistant. Think step-by-step. "
     "Use your search, wikipedia, and calculator tools whenever you need external or precise information."),
    ("placeholder", "{messages}")
])

# Use create_react_agent with the prompt parameter instead of system_prompt
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=prompt,  # ✅ Use 'prompt' instead of 'system_prompt'
    checkpointer=memory,
    debug=False  # Toggle to True if you want to inspect low-level LangGraph execution traces
)

# Configuration for managing conversation thread ID
config = {"configurable": {"thread_id": "jupyter_chat_session"}}

# ==========================================
# 4. Interactive Jupyter Function
# ==========================================
def ask_agent(user_input: str):
    """Call this function inside any Jupyter cell to talk to the agent."""
    if not user_input.strip():
        return
        
    print(f"You: {user_input}")
    print("Thinking...")
    
    try:
        # Pass the message payload structure expected by create_react_agent
        result = agent.invoke(
            {"messages": [{"role": "user", "content": user_input}]}, 
            config=config
        )
        
        # Pull the absolute last response generated in the message graph state
        ai_response = result["messages"][-1].content
        print(f"\nAI: {ai_response}\n")
        
    except Exception as e:
        print(f"\nExecution Error: {e}\n")

print("\n🚀 Setup Complete! You can now converse with the agent using the ask_agent() function.")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_14192\2435853348.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


🔄 Initializing Ollama (Llama 3.1)...
📦 Loading search and math tools...
🤖 Compiling ReAct Agent Graph...

🚀 Setup Complete! You can now converse with the agent using the ask_agent() function.


C:\Users\Administrator\AppData\Local\Temp\ipykernel_14192\2435853348.py:59: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [2]:
ask_agent("Who was Nikola Tesla?")

You: Who was Nikola Tesla?
Thinking...

AI: Nikola Tesla was a Serbian-American engineer, futurist, and inventor who made significant contributions to the design of the modern alternating current (AC) electricity supply system. He is known for his work on AC induction motors and related polyphase AC patents, which became the cornerstone of the polyphase system marketed by Westinghouse Electric. Tesla also experimented with wireless lighting and worldwide wireless electric power distribution, but was unable to complete his unfinished Wardenclyffe Tower project due to lack of funding. He is remembered as a pioneer in the field of electrical engineering and has been honored with the International System of Units (SI) measurement of magnetic flux density being named after him, the tesla.

